# APIM ❤️ VoiceLive

## Azure AI VoiceLive Audio lab
![flow](../../images/realtime-audio.gif)

Playground to try the APIM integration with an Azure AI Foundry VoiceLive resource for text and audio.

### Result
![result](result.png)

### Prerequisites

- [Python 3.12 or later version](https://www.python.org/) installed
- [VS Code](https://code.visualstudio.com/) installed with the [Jupyter notebook extension](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.jupyter) enabled
- [Python environment](https://code.visualstudio.com/docs/python/environments#_creating-environments) with the [requirements.txt](../../requirements.txt) or run `pip install -r requirements.txt` in your terminal
- [An Azure Subscription](https://azure.microsoft.com/free/) with [Contributor](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#contributor) + [RBAC Administrator](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#role-based-access-control-administrator) or [Owner](https://learn.microsoft.com/en-us/azure/role-based-access-control/built-in-roles/privileged#owner) roles
- [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli) installed and [Signed into your Azure subscription](https://learn.microsoft.com/cli/azure/authenticate-azure-cli-interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...


<a id='0'></a>
### 0️⃣ Initialize notebook variables

- Resources will be suffixed by a unique string based on your subscription id.
- Adjust the location parameters according your preferences and on the [product availability by Azure region.](https://azure.microsoft.com/explore/global-infrastructure/products-by-region/?cdn=disable&products=cognitive-services,api-management)
- Adjust the VoiceLive-compatible model and version according the [availability by region.](https://learn.microsoft.com/azure/ai-services/openai/concepts/models) 


In [1]:
import os, sys, json
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group_name = f"lab-{deployment_name}" # change the name to match your naming style
resource_group_location = "eastus2"

aiservices_config = [{"name": "foundry1", "location": "eastus2"}]

models_config = [{"name": "gpt-realtime", "publisher": "OpenAI", "version": "2025-08-28", "sku": "GlobalStandard", "capacity": 10}]

apim_sku = 'Basicv2'
apim_subscriptions_config = [{"name": "subscription1", "displayName": "Subscription 1"}]

inference_api_path = 'inference'  # path to the inference API in the APIM service
inference_api_type = "websocket"
inference_api_version = "2026-06-01-preview"
foundry_project_name = deployment_name

utils.print_ok('Notebook initialized')


✅ Notebook initialized ⌚ 06:56:16.016162 


<a id='1'></a>
### 1️⃣ Verify the Azure CLI and the connected Azure subscription

The following commands ensure that you have the latest version of the Azure CLI and that the Azure CLI is connected to your Azure subscription.

In [2]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

⚙️ Running: az account show 
✅ Retrieved az account ⌚ 06:56:22.312470 :5s]
👉🏽 Current user: jacwang@microsoft.com
👉🏽 Tenant ID: 16b3c013-d300-468d-ac64-7eda0820b6d3
👉🏽 Subscription ID: 6025ba02-1dfd-407f-b358-88f811c7c7aa


<a id='2'></a>
### 2️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declarative define all the resources that will be deployed in the specified resource group. Change the parameters or the [main.bicep](main.bicep) directly to try different configurations.

`openAIModelCapacity` is set intentionally low to `6` (6k tokens per minute) to trigger the retry logic in the load balancer (transparent to the user) as well as the priority failover from priority 1 to 2.

In [3]:
# Create the resource group if doesn't exist
utils.create_resource_group(resource_group_name, resource_group_location)

# Define the Bicep parameters
bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": { "value": apim_sku },
        "aiServicesConfig": { "value": aiservices_config },
        "modelsConfig": { "value": models_config },
        "apimSubscriptionsConfig": { "value": apim_subscriptions_config },
        "inferenceAPIPath": { "value": inference_api_path },
        "inferenceAPIType": { "value": inference_api_type },
        "foundryProjectName": { "value": foundry_project_name }
    }
}

# Write the parameters to the params.json file
with open('params.json', 'w') as bicep_parameters_file:
    bicep_parameters_file.write(json.dumps(bicep_parameters))

# Run the deployment
output = utils.run(f"az deployment group create --name {deployment_name} --resource-group {resource_group_name} --template-file main.bicep --parameters params.json",
    f"Deployment '{deployment_name}' succeeded", f"Deployment '{deployment_name}' failed")

⚙️ Running: az group show --name lab-realtime-audio 
👉🏽 Resource group lab-realtime-audio does not yet exist. Creating the resource group now...
⚙️ Running: az group create --name lab-realtime-audio --location eastus2 --tags source=ai-gateway 
✅ Resource group 'lab-realtime-audio' created ⌚ 06:56:37.595492 :7s]
⚙️ Running: az deployment group create --name realtime-audio --resource-group lab-realtime-audio --template-file main.bicep --parameters params.json 
✅ Deployment 'realtime-audio' succeeded ⌚ 06:58:55.694039 :18s]


<a id='3'></a>
### 3️⃣ Get the deployment outputs

Retrieve the required outputs from the Bicep deployment.

In [4]:
# Obtain all of the outputs from the deployment
output = utils.run(f"az deployment group show --name {deployment_name} -g {resource_group_name}", f"Retrieved deployment: {deployment_name}", f"Failed to retrieve deployment: {deployment_name}")

if output.success and output.json_data:
    log_analytics_id = utils.get_deployment_output(output, 'logAnalyticsWorkspaceId', 'Log Analytics Id')
    apim_service_id = utils.get_deployment_output(output, 'apimServiceId', 'APIM Service Id')
    apim_resource_gateway_url = utils.get_deployment_output(output, 'apimResourceGatewayURL', 'APIM API Gateway URL')
    apim_subscriptions = json.loads(utils.get_deployment_output(output, 'apimSubscriptions').replace("\'", "\""))
    for subscription in apim_subscriptions:
        subscription_name = subscription['name']
        subscription_key = subscription['key']
        utils.print_info(f"Subscription Name: {subscription_name}")
        utils.print_info(f"Subscription Key: ****{subscription_key[-4:]}")
    api_key = apim_subscriptions[0].get("key") # default api key to the first subscription key


⚙️ Running: az deployment group show --name realtime-audio -g lab-realtime-audio 
✅ Retrieved deployment: realtime-audio ⌚ 06:59:01.808639 :6s]
👉🏽 Log Analytics Id: 0d2fbfde-caa7-4044-9ad2-9db59e1c6fba
👉🏽 APIM Service Id: /subscriptions/6025ba02-1dfd-407f-b358-88f811c7c7aa/resourceGroups/lab-realtime-audio/providers/Microsoft.ApiManagement/service/apim-kdu56pasifuk4
👉🏽 APIM API Gateway URL: https://apim-kdu56pasifuk4.azure-api.net
👉🏽 Subscription Name: subscription1
👉🏽 Subscription Key: ****ba7f


<a id='4'></a>
### 4 Install Python library requirements

In [ ]:
%pip install -r requirements.txt

<a id='text'></a>
### 🧪 Test the VoiceLive API using just text

👉 Based on the Azure AI VoiceLive SDK samples.


In [ ]:
voice_live_endpoint = f"{apim_resource_gateway_url}/{inference_api_path}"
voice_live_model = models_config[0]['name']
masked_api_key = f"****{api_key[-4:]}" if api_key else "<missing>"

print(voice_live_endpoint)
print(voice_live_model)
print(inference_api_version)
print(masked_api_key)


In [ ]:
import asyncio, nest_asyncio
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import Modality, RequestSession, ServerEventType
from azure.core.credentials import AzureKeyCredential

nest_asyncio.apply()

async def main() -> None:
    async with connect(
        endpoint=voice_live_endpoint,
        credential=AzureKeyCredential(api_key),
        model=voice_live_model,
        api_version=inference_api_version,
    ) as connection:
        await connection.session.update(
            session=RequestSession(modalities=[Modality.TEXT])
        )
        await connection.conversation.item.create(
            item={
                "type": "message",
                "role": "user",
                "content": [{"type": "input_text", "text": "What is the capital of Portugal?"}],
            }
        )
        await connection.response.create()
        async for event in connection:
            if event.type == ServerEventType.RESPONSE_TEXT_DELTA:
                print(event.delta, flush=True, end="")
            elif event.type == ServerEventType.RESPONSE_TEXT_DONE:
                print()
            elif event.type == ServerEventType.RESPONSE_DONE:
                break
            elif event.type == ServerEventType.ERROR:
                raise RuntimeError(event.error.message)

if __name__ == "__main__":
    asyncio.run(main())


In [ ]:
!pip show azure-ai-voicelive


In [ ]:
import importlib.metadata
print(importlib.metadata.version("azure-ai-voicelive"))


In [ ]:
import base64
import asyncio
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import Modality, RequestSession, ServerEventType
from azure.core.credentials import AzureKeyCredential

async def main() -> None:
    """
    When prompted for user input, type a message and hit enter to send it to the model.
    Enter "q" to quit the conversation.
    """

    async with connect(
        endpoint=voice_live_endpoint,
        credential=AzureKeyCredential(api_key),
        model=voice_live_model,
        api_version=inference_api_version,
    ) as connection:
        await connection.session.update(
            session=RequestSession(modalities=[Modality.TEXT, Modality.AUDIO], voice="alloy")
        )
        while True:
            user_input = input("Enter a message: ")
            if user_input == "q":
                break

            await connection.conversation.item.create(
                item={
                    "type": "message",
                    "role": "user",
                    "content": [{"type": "input_text", "text": user_input}],
                }
            )
            await connection.response.create()
            async for event in connection:
                if event.type == ServerEventType.RESPONSE_TEXT_DELTA:
                    print(event.delta, flush=True, end="")
                elif event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                    audio_data = base64.b64decode(event.delta)
                    print(f"Received {len(audio_data)} bytes of audio data.")
                elif event.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DELTA:
                    print(f"Received text delta: {event.delta}")
                elif event.type == ServerEventType.RESPONSE_TEXT_DONE:
                    print()
                elif event.type == ServerEventType.RESPONSE_DONE:
                    break
                elif event.type == ServerEventType.ERROR:
                    raise RuntimeError(event.error.message)

asyncio.run(main())


<a id='fastrtc'></a>
### 🧪 Test the VoiceLive API using FastRTC + Gradio

⚡ FastRTC is an elegant realtime library communication library to enable you to easily and quickly build RTC application both using websockets and WebRTC.

Please ensure you have run the pip command succefully to install all required packages


In [ ]:
import asyncio, base64
import gradio as gr
import numpy as np
from azure.ai.voicelive.aio import connect
from azure.ai.voicelive.models import (
    AudioInputTranscriptionOptions,
    InputAudioFormat,
    Modality,
    OutputAudioFormat,
    RequestSession,
    ServerEventType,
    ServerVad,
)
from azure.core.credentials import AzureKeyCredential
from fastrtc import (
    AdditionalOutputs,
    AsyncStreamHandler,
    Stream,
    wait_for_item,
    UIArgs
)

SAMPLE_RATE = 24000

VOICE_LIVE_ENDPOINT = voice_live_endpoint
VOICE_LIVE_API_KEY = api_key
VOICE_LIVE_API_VERSION = inference_api_version
VOICE_LIVE_MODEL = voice_live_model
SESSION_CONFIG = RequestSession(
    input_audio_transcription=AudioInputTranscriptionOptions(model="whisper-1"),
    turn_detection=ServerVad(
        threshold=0.4,
        prefix_padding_ms=300,
        silence_duration_ms=600,
        interrupt_response=True,
    ),
    instructions="Your name is Amy. You're a helpful agent who responds initially with a calm British accent, but also can speak in any language as the user chooses to. Always start the conversation with a cheery hello",
    voice="alloy",
    modalities=[Modality.TEXT, Modality.AUDIO],
    input_audio_format=InputAudioFormat.PCM16,
    output_audio_format=OutputAudioFormat.PCM16,
)

class VoiceLiveHandler(AsyncStreamHandler):
    def __init__(self) -> None:
        super().__init__(
            expected_layout="mono",
            output_sample_rate=SAMPLE_RATE,
            output_frame_size=480,
            input_sample_rate=SAMPLE_RATE,
        )
        self.connection = None
        self.output_queue = asyncio.Queue()

    def copy(self):
        return VoiceLiveHandler()

    async def welcome(self):
        await self.connection.conversation.item.create(  # type: ignore[attr-defined]
            item={
                "type": "message",
                "role": "user",
                "content": [{"type": "input_text", "text": "what's your name?"}],
            }
        )
        await self.connection.response.create()  # type: ignore[attr-defined]

    async def start_up(self):
        """
        Establish a persistent realtime connection to the VoiceLive backend.
        The connection is configured for server-side Voice Activity Detection.
        """
        async with connect(
            endpoint=VOICE_LIVE_ENDPOINT,
            credential=AzureKeyCredential(VOICE_LIVE_API_KEY),
            model=VOICE_LIVE_MODEL,
            api_version=VOICE_LIVE_API_VERSION,
        ) as conn:
            await conn.session.update(session=SESSION_CONFIG)
            self.connection = conn

            # Uncomment the following line to send a welcome message to the assistant.
            # await self.welcome()

            async for event in self.connection:
                if event.type == ServerEventType.INPUT_AUDIO_BUFFER_SPEECH_STARTED:
                    self.clear_queue()
                elif event.type == ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED:
                    await self.output_queue.put(AdditionalOutputs(event))
                elif event.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DONE:
                    await self.output_queue.put(AdditionalOutputs(event))
                elif event.type == ServerEventType.RESPONSE_AUDIO_DELTA:
                    await self.output_queue.put(
                        (
                            self.output_sample_rate,
                            np.frombuffer(base64.b64decode(event.delta), dtype=np.int16).reshape(1, -1),
                        ),
                    )
                elif event.type == ServerEventType.ERROR:
                    raise RuntimeError(event.error.message)

    async def receive(self, frame: tuple[int, np.ndarray]) -> None:
        """
        Receives an audio frame from the stream and sends it into the realtime API.
        The audio data is encoded as Base64 before appending to the connection's input.
        """
        if not self.connection:
            return
        _, array = frame
        array = array.squeeze()
        audio_message = base64.b64encode(array.tobytes()).decode("utf-8")
        await self.connection.input_audio_buffer.append(audio=audio_message)

    async def emit(self) -> tuple[int, np.ndarray] | AdditionalOutputs | None:
        """
        Waits for and returns the next output from the output queue.
        The output may be an audio chunk or an additional output such as transcription.
        """
        return await wait_for_item(self.output_queue)

    async def shutdown(self) -> None:
        if self.connection:
            await self.connection.close()
            self.connection = None

def update_chatbot(chatbot: list[dict], content):
    """
    Append the completed transcription to the chatbot messages.
    """
    if content.type == ServerEventType.CONVERSATION_ITEM_INPUT_AUDIO_TRANSCRIPTION_COMPLETED:
        chatbot.append({"role": "user", "content": content.transcript})
    elif content.type == ServerEventType.RESPONSE_AUDIO_TRANSCRIPT_DONE:
        chatbot.append({"role": "assistant", "content": content.transcript})
    return chatbot

ui_args: UIArgs = UIArgs(
    title="APIM ❤️ VoiceLive - Contoso Assistant 🤖",
)
chatbot = gr.Chatbot(type="messages")
latest_message = gr.Textbox(type="text", visible=True)

stream = Stream(
    VoiceLiveHandler(),
    mode="send-receive",
    modality="audio",
    additional_inputs=[chatbot],
    additional_outputs=[chatbot],
    additional_outputs_handler=update_chatbot,
    ui_args=ui_args,
)

if __name__ == "__main__":
    stream.ui.launch(server_port=7990)


<a id='kql'></a>
### 🔍 Display model usage


In [ ]:
import pandas as pd

query = "model_usage"

output = utils.run(f"az monitor log-analytics query -w {log_analytics_id} --analytics-query \"{query}\"", "Retrieved log analytics query output", "Failed to retrieve log analytics query output") 
if output.success and output.json_data:
    table = output.json_data
    display(pd.DataFrame(table))


<a id='clean'></a>
### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.
Use the [clean-up notebook](clean-up-voicelive-resources.ipynb) for that.